<a href="https://colab.research.google.com/github/dtreku/mytestrepo/blob/main/openalex_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests, pandas as pd, time

def openalex_search_by_issn(keywords, issns=None, start_year=None, end_year=None,
                            per_page=200, total_results=500, polite_delay=2):
    base = "https://api.openalex.org/works"
    all_rows = []

    # --- Build filter string ---
    filters = []
    if start_year is not None and end_year is not None:
        filters.append(f"publication_year:{start_year}-{end_year}")
    if issns and isinstance(issns, (list, tuple)) and len(issns) > 0:
        filters.append(f"primary_location.source.issn:{'|'.join(issns)}")
    filter_str = ",".join(filters) if filters else None

    # --- Loop over keywords ---
    for kw in keywords:
        cursor = "*"
        fetched = 0
        while fetched < total_results:
            params = {
                "search": kw,
                "per-page": per_page,
                "cursor": cursor,
                "select": "id,display_name,doi,publication_year,authorships,"
                          "cited_by_count,primary_location,abstract_inverted_index,concepts"
            }
            if filter_str:
                params["filter"] = filter_str

            r = requests.get(base, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()

            for item in data.get("results", []):
                # ✅ safer extraction of venue information
                src = None
                if item.get("primary_location") and isinstance(item["primary_location"], dict):
                    src = item["primary_location"].get("source") or {}

                concepts = item.get("concepts", [])
                concept_names = [c.get("display_name") for c in concepts if c.get("display_name")]

                journal_name = src.get("display_name") if isinstance(src, dict) else None
                issn_val = src.get("issn_l") if isinstance(src, dict) else None

                all_rows.append({
                    "keyword": kw,
                    "id": item.get("id"),
                    "title": item.get("display_name"),
                    "doi": item.get("doi"),
                    "year": item.get("publication_year"),
                    "journal": journal_name,
                    "issn": issn_val,
                    "authors": ", ".join(
                        a.get("author", {}).get("display_name", "")
                        for a in item.get("authorships", [])
                    ),
                    "concepts": ", ".join(concept_names) if concept_names else None,
                    "cited_by": item.get("cited_by_count"),
                    "abstract_inverted_index": item.get("abstract_inverted_index"),
                })

            fetched += len(data.get("results", []))
            cursor = (data.get("meta") or {}).get("next_cursor")
            if not cursor:
                break
            time.sleep(polite_delay)

    return pd.DataFrame(all_rows)


In [ ]:
def reconstruct_abstract(inv_idx):
    """Safely convert OpenAlex's abstract_inverted_index dict to a plain string."""
    if not isinstance(inv_idx, dict):
        return None

    try:
        # Find the max index in the inverted index
        max_index = max(max(v) for v in inv_idx.values())
        tokens = [""] * (max_index + 1)

        for word, positions in inv_idx.items():
            for pos in positions:
                # Replace missing or None with empty string instead of None
                if pos < len(tokens):
                    tokens[pos] = word or ""

        # Filter out any residual empty strings and join cleanly
        return " ".join(t for t in tokens if t)
    except Exception as e:
        print(f"⚠️ Error reconstructing abstract: {e}")
        return None


In [ ]:
# list all terms you are interested in here. see examples
query_terms = [
    # Core fairness/bias
    '"algorithmic fairness"', '"algorithmic bias"', '"AI fairness"', '"AI bias"',
    '"AI discrimination"', '"algorithmic discrimination"',
    '"machine learning fairness"', '"machine learning bias"', '"machine learning discrimination"',
]


In [ ]:
# default argument is set to none. see function in first cell. below is a list of IS journals
issns = [
    # AIS Basket of 11
    "0960-085X",  # EJIS
    "0378-7206",  # Information & Management
    "1350-1917",  # Information Systems Journal
    "1047-7047",  # Information Systems Research
    "1536-9323",  # JAIS
    "0268-3962",  # Journal of Information Technology
    "0742-1222",  # JMIS
    "0963-8687",  # JSIS
    "0276-7783",  # MIS Quarterly
    "1387-3326",  # Information Systems Frontiers
    "1529-3181",  # Communications of the AIS
]

# Example:
df = openalex_search_by_issn(
    keywords=query_terms,
    start_year=2000,
    end_year=2025,
    total_results=20000
)